# PreventIA - Clasificación de daños con una MLP en Google Colab
Ejecuta las celdas en orden. Conecta la cuenta de Google que tiene el dataset o su acceso directo en Mi unidad.

Ruta del dataset: `Mi unidad/United_States/United_States/`, con `train` y `test` dentro.
Si compartes este notebook, cada compañero conecta su cuenta y ejecuta su propio entrenamiento.

El proceso conserva la lectura de imágenes y XML, el preprocesamiento y una sola MLP.
Al final puedes **subir una fotografía, marcar regiones en el navegador y pulsar Terminar para clasificarlas**.
Las cuatro clases son D00 (longitudinal), D10 (transversal), D20 (cocodrilo) y D40 (bache).
La selección es manual y no existe una clase «sin daño».


## 1. Lectura del dataset
Trabajamos con RDD2022 de Estados Unidos. Las fotografías están en `train/images` y
sus anotaciones en `train/annotations/xmls`. Cada XML contiene la clase y las coordenadas
del daño. Una fotografía puede contener varios daños.

Primero contamos los archivos y verificamos que cada imagen de entrenamiento tenga su XML.
Las fotografías de `test` se cuentan y se pueden usar para la demostración; al no tener
XML locales, no se utilizan para calcular el porcentaje de aciertos.

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import hashlib
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

CLASES = {'D00': 'Grieta longitudinal', 'D10': 'Grieta transversal',
          'D20': 'Grieta tipo cocodrilo', 'D40': 'Bache'}
# Conectar Drive y localizar el dataset compartido.
from google.colab import drive
drive.mount('/content/drive')

DATOS = Path('/content/drive/MyDrive/United_States/United_States')
print('Ruta utilizada:', DATOS)
if not (DATOS / 'train' / 'annotations' / 'xmls').is_dir():
    raise FileNotFoundError(
        'No se encontró el dataset. Revisa la cuenta conectada y la ruta. '
        'Si te compartieron la carpeta, añade un acceso directo de la carpeta exterior '
        'United_States directamente a Mi unidad.'
    )
print('Dataset localizado.')


In [ ]:
fotos_train = sorted(p for p in (DATOS / 'train' / 'images').iterdir()
                     if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
xmls = sorted((DATOS / 'train' / 'annotations' / 'xmls').glob('*.xml'))
fotos_test = sorted(p for p in (DATOS / 'test' / 'images').glob('*')
                    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
sin_xml = {p.stem for p in fotos_train} - {p.stem for p in xmls}
sin_foto = {p.stem for p in xmls} - {p.stem for p in fotos_train}
print('Fotografías de train:', len(fotos_train))
print('Anotaciones XML:', len(xmls))
print('Fotografías de test:', len(fotos_test))
print('Fotografías sin XML:', len(sin_xml), '| XML sin fotografía:', len(sin_foto))
if sin_xml or sin_foto or not xmls:
    raise ValueError('Revisa la correspondencia de las imágenes y los XML.')

ejemplo_xml = ET.parse(xmls[0]).getroot()
print('Ejemplo de XML:', xmls[0].name)
for objeto in ejemplo_xml.findall('object'):
    clase = objeto.findtext('name')
    coordenadas = [objeto.findtext(f'bndbox/{k}') for k in ('xmin', 'ymin', 'xmax', 'ymax')]
    print('Daño:', clase, '| Rectángulo:', coordenadas)

## 2. Preprocesamiento de las imágenes
Por cada daño anotado:
1. Recortamos la región indicada en el XML, comprobando que sus coordenadas sean válidas.
2. Convertimos el recorte a **escala de grises** y lo redimensionamos a **32 × 32 píxeles**.
3. Dividimos los píxeles entre **255** para obtener valores entre 0 y 1.
4. Ordenamos los píxeles en una lista de **1 024 números**, que será la entrada de la MLP.

La etiqueta del XML se guarda en `y`; los píxeles, en `X`. Las fotos idénticas se identifican
para mantenerlas juntas al dividir los datos. No generamos carpetas de recortes ni CSV.

In [ ]:
def preparar_recorte(imagen):
    gris = imagen.convert('L').resize((32, 32), Image.Resampling.BILINEAR)
    return np.asarray(gris, dtype=np.float32).reshape(-1) / 255.0

X, y, grupos, ejemplos = [], [], [], []
for archivo in xmls:
    anotacion = ET.parse(archivo).getroot()
    ruta = DATOS / 'train' / 'images' / anotacion.findtext('filename')
    with Image.open(ruta) as original:
        imagen = original.convert('RGB')
        grupo = hashlib.sha256(imagen.tobytes()).hexdigest()
        for objeto in anotacion.findall('object'):
            clase = objeto.findtext('name')
            if clase not in CLASES:
                raise ValueError(f'Clase desconocida en {archivo.name}: {clase}')
            caja = tuple(int(objeto.findtext(f'bndbox/{k}'))
                         for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            x1, y1, x2, y2 = caja
            if not (0 <= x1 < x2 <= imagen.width and 0 <= y1 < y2 <= imagen.height):
                raise ValueError(f'Revisa las coordenadas en {archivo.name}')
            X.append(preparar_recorte(imagen.crop(caja)))
            y.append(clase)
            grupos.append(grupo)
            ejemplos.append((ruta, caja))

X, y, grupos = np.array(X), np.array(y), np.array(grupos)
print('Total de recortes preparados:', len(y))
print('Entradas por recorte:', X.shape[1])
print('Valores de píxel:', float(X.min()), 'a', float(X.max()))
print('Daños anotados por clase (no son cantidades de fotografías):')
for clase, nombre in CLASES.items():
    print(f'  {clase} — {nombre}: {int(np.sum(y == clase))}')

In [ ]:
# Ejemplo visual: fotografía → región anotada → entrada preparada.
from matplotlib.patches import Rectangle
ruta, caja = ejemplos[0]
with Image.open(ruta) as archivo:
    foto_ejemplo = archivo.convert('RGB')
x1, y1, x2, y2 = caja
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
ejes[0].imshow(foto_ejemplo)
ejes[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=2))
ejes[0].set_title('Fotografía y caja del XML')
ejes[1].imshow(foto_ejemplo.crop(caja))
ejes[1].set_title(f'Recorte original: {y[0]}')
ejes[2].imshow(X[0].reshape(32, 32), cmap='gray', vmin=0, vmax=1)
ejes[2].set_title('Grises · 32 × 32 · valores 0–1')
for eje in ejes:
    eje.axis('off')
plt.tight_layout()
plt.show()

## 3. Separar entrenamiento y validación
Reservamos aproximadamente el **80 % para entrenar y el 20 % para validar**.
Los recortes de una foto y las fotos idénticas permanecen en el mismo grupo.
La semilla 42 permite repetir la división. Ambos conjuntos salen de `train`.
La proporción de recortes puede variar, porque cada foto contiene una cantidad diferente de daños.

In [ ]:
g_train, g_val = train_test_split(np.unique(grupos), test_size=0.2, random_state=42)
es_train = np.isin(grupos, g_train)
X_train, X_val = X[es_train], X[~es_train]
y_train, y_val = y[es_train], y[~es_train]
ejemplos_val = [ejemplos[i] for i in np.flatnonzero(~es_train)]
print('Recortes para entrenar:', len(y_train))
print('Recortes para validar:', len(y_val))
print('Entradas por recorte:', X.shape[1])

## 4. Entrenar y comprobar la MLP
Utilizamos **una capa oculta de 32 neuronas** y cuatro clases de salida.
`fit` aprende con los ejemplos de entrenamiento y `predict` clasifica los de validación.
El entrenamiento puede tardar unos minutos. Mostramos únicamente el número de aciertos
y su porcentaje global; las clases menos frecuentes pueden tener menor rendimiento.

In [ ]:
modelo = MLPClassifier(hidden_layer_sizes=(32,), activation='relu',
                       solver='adam', alpha=0.01, max_iter=300, random_state=42)
modelo.fit(X_train, y_train)
print('Entrenamiento terminado.')

In [ ]:
predicciones = modelo.predict(X_val)
aciertos = int(np.sum(predicciones == y_val))
print(f'Aciertos: {aciertos} de {len(y_val)}')
print(f'Porcentaje de aciertos: {aciertos / len(y_val):.2%}')

## 5. Probar un ejemplo conocido
Cambia `numero` para visualizar otro recorte de validación.
La etiqueta real se muestra para comprobar la predicción y no se entrega como entrada a la red.

In [ ]:
numero = 0  # Cambia este número para probar otro ejemplo.
if not 0 <= numero < len(X_val):
    raise ValueError(f'Elige un número entre 0 y {len(X_val) - 1}.')
clase = str(modelo.predict(X_val[numero:numero + 1])[0])
print('Predicción:', clase, '—', CLASES[clase])
print('Etiqueta real:', y_val[numero], '—', CLASES[y_val[numero]])
ruta, caja = ejemplos_val[numero]
with Image.open(ruta) as foto:
    plt.imshow(foto.convert('RGB').crop(caja))
plt.title(f'Predicción: {clase} | Real: {y_val[numero]}')
plt.axis('off')
plt.show()

## 6. Subir una fotografía y marcar daños en Colab
Ejecuta las dos celdas siguientes después de entrenar la red.
1. En la última celda aparecerá el botón para elegir una fotografía de tu computadora.
2. Arrastra sobre la imagen para marcar una o varias regiones dañadas.
3. Usa **Borrar selecciones** para empezar de nuevo o **Cancelar** para salir.
4. Pulsa **Terminar** para obtener las predicciones y la fotografía anotada.

La celda permanece en ejecución mientras espera tus selecciones: es normal.
Para probar otra fotografía, ejecuta otra vez la última celda.

No se requiere XML para esta prueba. Las predicciones no forman parte del porcentaje de aciertos.
Si marcas pavimento sano, la red también asignará una de las cuatro clases de daño.


In [ ]:
from google.colab import files, output
from io import BytesIO
import base64
import json


def abrir_y_marcar():
    if 'modelo' not in globals() or not hasattr(modelo, 'classes_'):
        raise ValueError('Ejecuta primero las celdas de preparación y entrenamiento.')

    print('Elige una fotografía. Luego marca sus regiones y pulsa Terminar.')
    subidos = files.upload()
    if not subidos:
        return None
    if len(subidos) != 1:
        raise ValueError('Selecciona una sola fotografía y vuelve a ejecutar esta celda.')
    with Image.open(BytesIO(next(iter(subidos.values())))) as archivo:
        foto = archivo.convert('RGB')

    # La copia pequeña es solo para mostrar; se recorta la foto original.
    vista = foto.copy()
    vista.thumbnail((900, 650))
    buffer = BytesIO()
    vista.save(buffer, format='PNG')
    imagen = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()

    # Este bloque dibuja el selector en el navegador; no entrena otro modelo.
    selector = r'''
    (async () => {
        const panel = document.createElement('div');
        panel.innerHTML = `<p>Arrastra para marcar uno o varios daños. Después pulsa Terminar.</p>
            <canvas style="max-width:100%;height:auto;touch-action:none;cursor:crosshair"></canvas>
            <p class="estado">0 regiones seleccionadas</p>
            <button class="borrar">Borrar selecciones</button>
            <button class="terminar">Terminar</button>
            <button class="cancelar">Cancelar</button>`;
        const foto = new Image();
        await new Promise((resolve, reject) => {
            foto.onload = resolve;
            foto.onerror = () => reject(new Error('No se pudo mostrar la imagen.'));
            foto.src = __IMAGEN__;
        });
        document.body.appendChild(panel);
        const lienzo = panel.querySelector('canvas');
        lienzo.width = foto.width; lienzo.height = foto.height;
        const ctx = lienzo.getContext('2d');
        let cajas = [], inicio = null;
        const punto = e => {
            const r = lienzo.getBoundingClientRect();
            return [Math.max(0, Math.min(lienzo.width, (e.clientX-r.left)*lienzo.width/r.width)),
                    Math.max(0, Math.min(lienzo.height, (e.clientY-r.top)*lienzo.height/r.height))];
        };
        const caja = fin => [Math.min(inicio[0], fin[0]), Math.min(inicio[1], fin[1]),
                            Math.max(inicio[0], fin[0]), Math.max(inicio[1], fin[1])];
        const dibujar = (activa = null) => {
            ctx.drawImage(foto, 0, 0);
            ctx.strokeStyle = 'red'; ctx.fillStyle = 'red';
            ctx.lineWidth = 2; ctx.font = 'bold 16px Arial';
            [...cajas, ...(activa ? [activa] : [])].forEach(([x1,y1,x2,y2], i) => {
                ctx.strokeRect(x1, y1, x2-x1, y2-y1);
                ctx.fillText(String(i+1), x1+4, y1+18);
            });
            panel.querySelector('.estado').textContent = cajas.length + ' regiones seleccionadas';
        };
        lienzo.onpointerdown = e => {
            if (e.button !== 0 || inicio) return;
            inicio = punto(e); lienzo.setPointerCapture(e.pointerId);
        };
        lienzo.onpointermove = e => { if (inicio) dibujar(caja(punto(e))); };
        lienzo.onpointerup = e => {
            if (!inicio) return;
            const c = caja(punto(e)); inicio = null;
            if (c[2]-c[0] >= 5 && c[3]-c[1] >= 5) cajas.push(c);
            dibujar();
        };
        lienzo.onpointercancel = () => { inicio = null; dibujar(); };
        panel.querySelector('.borrar').onclick = () => { cajas = []; inicio = null; dibujar(); };
        dibujar();
        if (typeof google !== 'undefined' && google.colab && google.colab.output) {
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        }
        return await new Promise(resolve => {
            panel.querySelector('.terminar').onclick = () => { panel.remove(); resolve(cajas); };
            panel.querySelector('.cancelar').onclick = () => { panel.remove(); resolve([]); };
        });
    })()
    '''
    cajas = output.eval_js(selector.replace('__IMAGEN__', json.dumps(imagen)))

    resultados = []
    for x1, y1, x2, y2 in cajas:
        caja = (round(x1 * foto.width / vista.width),
                round(y1 * foto.height / vista.height),
                round(x2 * foto.width / vista.width),
                round(y2 * foto.height / vista.height))
        entrada = preparar_recorte(foto.crop(caja)).reshape(1, -1)
        clase = str(modelo.predict(entrada)[0])
        resultados.append((caja, clase))
    return foto, resultados


In [ ]:
from matplotlib.patches import Rectangle

seleccion = abrir_y_marcar()
if seleccion and seleccion[1]:
    foto, resultados = seleccion
    fig, eje = plt.subplots(figsize=(8, 8))
    eje.imshow(foto)
    for numero, (caja, clase) in enumerate(resultados, 1):
        x1, y1, x2, y2 = caja
        eje.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=2))
        eje.text(x1, y1, f'{numero}: {clase}', color='white', backgroundcolor='red')
        print(f'Región {numero}: {clase} — {CLASES[clase]}')
    eje.set_title('Predicciones en las regiones seleccionadas')
    eje.axis('off')
    plt.show()
else:
    print('No se seleccionaron regiones.')

El modelo permanece en memoria mientras dura la sesión de Colab. Si se reinicia o se pierde la sesión,
vuelve a conectar Drive y ejecuta la preparación y el entrenamiento. Las imágenes subidas para probar
se guardan temporalmente en el entorno de Colab; no se guardan automáticamente en Drive.

La evaluación mide clasificación de recortes anotados, no localización automática de daños.
Los recortes de fotos idénticas permanecen juntos, sin garantizar independencia por carretera.
